# 3 · From the failures to new improvements

Notebook 2 ended with a flat aggregation axis. This notebook does what the failure
analysis licenses: it reads the mechanism, asks what it *allows*, and builds new variants
from measurements rather than from intuition.

**What the mechanism says.** Global bottom-*k%* selection is already a well-targeted
anomaly search. Any scheme that forces the quota to be spent locally is strictly worse.
So a variant can only help if it:

1. keeps global selection, and changes how the selected scores are combined; or
2. changes *how many* scores are selected, adaptively; or
3. targets something other than ranking — a threshold, or compute.

In [1]:
import os, sys, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS, RES, FIG = ROOT/"scripts", ROOT/"results", ROOT/"results"/"figures"
sys.path.insert(0, str(SCRIPTS))
import minkpp as M
_s = importlib.util.spec_from_file_location("cachemod", SCRIPTS/"01_cache.py")
cachemod = importlib.util.module_from_spec(_s); _s.loader.exec_module(cachemod)

MODEL = os.environ.get("MINKPP_MODEL", "mamba-1.4b-hf")   # the paper's target model
SPLITS = ["WikiMIA_length32", "WikiMIA_length64", "WikiMIA_length128"]
pd.set_option("display.width", 120)
print("library self-check:", "PASS" if M.self_check(verbose=False) else "FAIL")


library self-check: PASS


In [2]:
paired = pd.read_csv(RES/f"paired_{MODEL}.csv")
auroc  = pd.read_csv(RES/f"auroc_{MODEL}.csv")
new = paired[paired.family.str.startswith("N")]
new[["split","variant","family","delta","lo","hi","p_gt0"]].round(2)

,split,variant,family,delta,lo,hi,p_gt0
46,WikiMIA_length32,n1_kens,N1 k-ensemble,0.21,-0.12,0.56,0.89
47,WikiMIA_length32,n2_adaptk,N2 length-adaptive k,0.39,-0.64,1.41,0.76
48,WikiMIA_length32,n3_rank,N3 rank-normalised,-17.74,-23.02,-12.05,0.00
49,WikiMIA_length32,n4_trim0.1,N4 trimmed tail,-0.52,-1.68,0.68,0.19
50,WikiMIA_length32,n4_trim0.25,N4 trimmed tail,-1.69,-3.85,0.36,0.06
97,WikiMIA_length64,n1_kens,N1 k-ensemble,-0.11,-0.49,0.28,0.30
98,WikiMIA_length64,n2_adaptk,N2 length-adaptive k,-0.45,-1.67,0.77,0.22
99,WikiMIA_length64,n3_rank,N3 rank-normalised,-19.27,-25.88,-12.61,0.00
100,WikiMIA_length64,n4_trim0.1,N4 trimmed tail,-0.99,-2.56,0.64,0.11
101,WikiMIA_length64,n4_trim0.25,N4 trimmed tail,-3.86,-6.45,-1.25,0.00


## 3.1 The four derived variants

| | Derived from | What it does |
|---|---|---|
| **N1** k-ensemble | §2.5 — k moves AUROC more than any aggregation variant, and the paper never says which k it used | average the statistic over a grid of k instead of picking one |
| **N2** length-adaptive k | §2.4 — the per-item fixed-count gain correlates with token count at r ≈ −0.7 | let k shrink with passage length, the continuous form of that observation |
| **N3** rank-normalised | the suspicion that longer items have heavier absolute-z tails | aggregate within-item ranks rather than z values |
| **N4** trimmed tail | the mechanism says selection is well targeted — so are its *worst* members outliers? | drop the most extreme selected tokens before averaging |

N1 and N2 are the honest ones: each is a direct formalisation of a measurement already in
hand, not a new idea.

## 3.2 Did they work?

In [3]:
summ = (new.groupby("variant")
        .agg(mean_delta=("delta","mean"), best=("delta","max"),
             worst=("delta","min"),
             any_sig=("significant","any"))
        .round(2).sort_values("mean_delta", ascending=False))
summ

,mean_delta,best,worst,any_sig
variant,,,,
n1_kens,0.03,0.21,-0.11,False
n2_adaptk,-0.56,0.39,-1.60,False
n4_trim0.1,-1.06,-0.52,-1.67,False
n4_trim0.25,-2.48,-1.69,-3.86,True
n3_rank,-17.36,-15.07,-19.27,True


### N3 fails hardest, and the reason is instructive

Replacing z by its within-item rank throws away *how far* below the mode a token sits and
keeps only the ordering. But the ordering inside an item is the same for members and
non-members — every item has a lowest token. The discriminative information was in the
**magnitude**, and rank-normalisation deletes exactly that. This is a clean example of a
variant that removes a nuisance and the signal together.

### N1 and N2 are null, not harmful

Both sit within noise. For N1 that is expected in hindsight: averaging over k is a
smoothing operation, and the k-curve is nearly flat within a split, so there is little to
smooth. For N2 the per-item mechanism was real (r ≈ −0.7) but the two halves cancel in
aggregate, exactly as fixed-count selection did.

## 3.3 H7, tested where it can actually show

H7 — length-conditional calibration — cannot change AUROC, because AUROC is
threshold-free: any monotone, item-independent recalibration leaves the ranking alone.
It can only matter at a **fixed operating point**, where a single threshold has to serve
items of different lengths. So we test it on TPR@5%FPR, and we test the premise first:
does the baseline score depend on length at all?

In [4]:
sp = "WikiMIA_length64"
df = pd.read_csv(RES/f"scores_{MODEL}_{sp}.csv")
r = np.corrcoef(df["minkpp_0.2"], df["n_tokens"])[0,1]
print(f"corr(baseline score, token count) = {r:+.3f}")

# length-conditional calibration: z-score the baseline within length terciles
q = pd.qcut(df["n_tokens"], 3, labels=False, duplicates="drop")
cal = df["minkpp_0.2"].copy()
for g in np.unique(q):
    m = q == g
    cal[m] = (cal[m] - cal[m].mean()) / (cal[m].std() + 1e-9)

y = df["label"].to_numpy()
for nm, s in [("baseline", df["minkpp_0.2"].to_numpy()), ("H7 calibrated", cal.to_numpy())]:
    print(f"{nm:16s} AUROC {100*M.auroc(y,s):.1f}   TPR@5%FPR {100*M.tpr_at_fpr(y,s):.1f}")

corr(baseline score, token count) = -0.037
baseline         AUROC 67.0   TPR@5%FPR 14.1
H7 calibrated    AUROC 67.0   TPR@5%FPR 12.7


If the correlation is weak, the premise of H7 is weak on this benchmark and the
calibration has little to correct — which is the result rather than a failure of the
implementation.

## 3.4 H10, read as a compute result rather than an accuracy one

H10 scores only a prefix of each item. As an accuracy improvement it is not expected to
win — less information cannot help. Its value is the **accuracy/compute trade-off**: how
much of the sequence is actually needed?

In [5]:
h10 = auroc[auroc.variant.str.startswith("h10_")].copy()
h10["frac"] = h10.variant.str.extract(r"frac(0\.\d+)").astype(float)
base = auroc[auroc.variant=="minkpp_0.2"][["split","auroc"]].rename(
        columns={"auroc":"base"})
h10 = h10.merge(base, on="split")
h10["retained_pct"] = (100*h10.auroc/h10.base).round(1)
h10[["split","frac","auroc","base","retained_pct"]].round(1).sort_values(["split","frac"])

,split,frac,auroc,base,retained_pct
6,WikiMIA_length128,0.2,70.9,67.9,104.4
7,WikiMIA_length128,0.5,68.6,67.9,101.0
8,WikiMIA_length128,0.8,67.9,67.9,100.0
0,WikiMIA_length32,0.2,60.1,66.4,90.5
1,WikiMIA_length32,0.5,64.7,66.4,97.4
2,WikiMIA_length32,0.8,66.5,66.4,100.1
3,WikiMIA_length64,0.2,64.1,67.0,95.7
4,WikiMIA_length64,0.5,65.7,67.0,98.2
5,WikiMIA_length64,0.8,65.8,67.0,98.2


Reading this as "how much accuracy survives at what fraction of the tokens" turns a
failed accuracy hypothesis into a usable efficiency statement — which is what §4.4 of the
regulations means by a lower-cost variant on an accuracy/compute trade-off.

## 3.5 What would have to change for the aggregation axis to pay

The mechanism is specific, so it predicts its own exceptions. Aggregation can only help
when the global bottom-*k%* is **spread across unrelated material**, i.e. when:

- passages are much longer than WikiMIA's 272-token maximum, so that one memorised span
  is a small fraction of the whole; or
- the passage is heterogeneous by construction — concatenated documents, a memorised
  quotation inside original text. The `WikiMIA_concat` split published by the same authors
  is built exactly this way and is the natural next test.

Neither is a rescue of the hypothesis on *this* benchmark. Both are falsifiable
predictions the mechanism makes, which is what distinguishes an explained negative result
from an unexplained one.